In [1]:
import requests
import pandas as pd
import sqlite3
from datetime import datetime

def fetch_weather():
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": 29.87,
        "longitude": 77.89,
        "current_weather": True
    }
    response = requests.get(url, params=params)
    data = response.json()
    current = data['current_weather']

    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "temperature": current['temperature'],
        "windspeed": current['windspeed'],
        "weathercode": current['weathercode'],
        "city": "Roorkee"
    }
    return pd.DataFrame([row])

def save_to_db(df, table_name):
    conn = sqlite3.connect("data/city_pulse.db")
    df.to_sql(table_name, conn, if_exists="append", index=False)
    conn.close()

df_weather = fetch_weather()
save_to_db(df_weather, "weather_logs")
print("✅ Weather saved!")
df_weather

✅ Weather saved!


,timestamp,temperature,windspeed,weathercode,city
0,2026-06-20 15:05:26,35.9,15.4,0,Roorkee


In [6]:
import requests
import pandas as pd
import sqlite3
from datetime import datetime

# ─── STEP 1: FETCH AIR QUALITY ───────────────────────────
def fetch_air_quality():
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": 29.87,
        "longitude": 77.89,
        "hourly": "pm10,pm2_5,carbon_monoxide,nitrogen_dioxide"
    }
    response = requests.get(url, params=params)
    data = response.json()

    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "pm2_5": data['hourly']['pm2_5'][0],
        "pm10":  data['hourly']['pm10'][0],
        "carbon_monoxide": data['hourly']['carbon_monoxide'][0],
        "nitrogen_dioxide": data['hourly']['nitrogen_dioxide'][0],
        "city": "Roorkee"
    }
    return pd.DataFrame([row])

df_air = fetch_air_quality()
print("✅ Air quality fetched!")
print(df_air)

# ─── STEP 2: CALCULATE AQI ───────────────────────────────
def calculate_aqi(pm2_5, pm10):
    # PM2.5 sub-index
    if pm2_5 <= 30:
        aqi_pm25 = (50/30) * pm2_5
    elif pm2_5 <= 60:
        aqi_pm25 = 50 + ((50/30) * (pm2_5 - 30))
    elif pm2_5 <= 90:
        aqi_pm25 = 100 + ((100/30) * (pm2_5 - 60))
    elif pm2_5 <= 120:
        aqi_pm25 = 200 + ((100/30) * (pm2_5 - 90))
    elif pm2_5 <= 250:
        aqi_pm25 = 300 + ((100/130) * (pm2_5 - 120))
    else:
        aqi_pm25 = 400 + ((100/130) * (pm2_5 - 250))

    # PM10 sub-index
    if pm10 <= 50:
        aqi_pm10 = pm10
    elif pm10 <= 100:
        aqi_pm10 = 50 + (pm10 - 50)
    elif pm10 <= 250:
        aqi_pm10 = 100 + ((100/150) * (pm10 - 100))
    elif pm10 <= 350:
        aqi_pm10 = 200 + ((100/100) * (pm10 - 250))
    elif pm10 <= 430:
        aqi_pm10 = 300 + ((100/80) * (pm10 - 350))
    else:
        aqi_pm10 = 400 + ((100/80) * (pm10 - 430))

    return round(max(aqi_pm25, aqi_pm10))

def get_aqi_level(aqi):
    if aqi <= 50:   return "Good 🟢"
    elif aqi <= 100: return "Satisfactory 🟡"
    elif aqi <= 200: return "Moderate 🟠"
    elif aqi <= 300: return "Poor 🔴"
    elif aqi <= 400: return "Very Poor 🟣"
    else:            return "Severe ⚫"

# Get values from df_air
pm2_5_value = df_air['pm2_5'].iloc[0]
pm10_value  = df_air['pm10'].iloc[0]

aqi   = calculate_aqi(pm2_5_value, pm10_value)
level = get_aqi_level(aqi)

# Add AQI to dataframe
df_air['aqi']       = aqi
df_air['aqi_level'] = level

# ─── STEP 3: SAVE TO DATABASE ────────────────────────────
conn = sqlite3.connect("data/city_pulse.db")
df_air.to_sql("air_quality_logs", conn, if_exists="replace", index=False)
conn.close()

# ─── STEP 4: PRINT RESULTS ───────────────────────────────
print("\n📍 Roorkee Air Quality Report")
print("─────────────────────────────")
print(f"PM2.5     : {pm2_5_value} µg/m³")
print(f"PM10      : {pm10_value} µg/m³")
print(f"AQI Score : {aqi}")
print(f"AQI Level : {level}")
print("\n✅ Saved to database!")
df_air

✅ Air quality fetched!
             timestamp  pm2_5   pm10  carbon_monoxide  nitrogen_dioxide  \
0  2026-06-20 15:25:48   87.3  106.9            632.0              46.8   

      city  
0  Roorkee  

📍 Roorkee Air Quality Report
─────────────────────────────
PM2.5     : 87.3 µg/m³
PM10      : 106.9 µg/m³
AQI Score : 191
AQI Level : Moderate 🟠

✅ Saved to database!


,timestamp,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,city,aqi,aqi_level
0,2026-06-20 15:25:48,87.3,106.9,632.0,46.8,Roorkee,191,Moderate 🟠


In [ ]:
from textblob import TextBlob

# Real world: Swiggy reads reviews → auto tags positive/negative
# We: Read citizen complaints → find positive/negative

complaints = [
    "Water supply has been cut for 3 days in sector 5",
    "Roads are excellent after repair, very smooth now",
    "Garbage not collected since Monday, very bad smell",
    "Street lights working perfectly, feel safe at night",
    "Electricity goes off every evening for 2 hours",
    "New park is beautiful, children love playing there",
    "Drainage system is blocked, water flooding the road",
    "Bus service is very punctual and clean"
]

rows = []
for text in complaints:
    blob = TextBlob(text)
    score = blob.sentiment.polarity

    if score > 0.1:
        label = "Positive"
    elif score < -0.1:
        label = "Negative"
    else:
        label = "Neutral"

    rows.append({
        "complaint": text,
        "sentiment_score": round(score, 2),
        "sentiment_label": label
    })

df_complaints = pd.DataFrame(rows)
save_to_db(df_complaints, "complaints_analysis")
print("✅ Complaints analyzed!")
df_complaints


In [ ]:
!pip install textblob
!python -m textblob.download_corpora

In [ ]:
from textblob import TextBlob

# Real world: Swiggy reads reviews → auto tags positive/negative
# We: Read citizen complaints → find positive/negative

complaints = [
    "Water supply has been cut for 3 days in sector 5",
    "Roads are excellent after repair, very smooth now",
    "Garbage not collected since Monday, very bad smell",
    "Street lights working perfectly, feel safe at night",
    "Electricity goes off every evening for 2 hours",
    "New park is beautiful, children love playing there",
    "Drainage system is blocked, water flooding the road",
    "Bus service is very punctual and clean"
]

rows = []
for text in complaints:
    blob = TextBlob(text)
    score = blob.sentiment.polarity

    if score > 0.1:
        label = "Positive"
    elif score < -0.1:
        label = "Negative"
    else:
        label = "Neutral"

    rows.append({
        "complaint": text,
        "sentiment_score": round(score, 2),
        "sentiment_label": label
    })

df_complaints = pd.DataFrame(rows)
save_to_db(df_complaints, "complaints_analysis")
print("✅ Complaints analyzed!")
df_complaints

In [ ]:
!pip install textblob scikit-learn statsmodels plotly pandas requests openpyxl streamlit


In [ ]:
from sklearn.ensemble import IsolationForest
import random

random.seed(42)

# Simulate 30 days normal weather
rows = []
for i in range(30):
    rows.append({
        "timestamp": f"2024-06-{i+1:02d}",
        "temperature": round(random.uniform(28, 38), 1),
        "windspeed": round(random.uniform(5, 20), 1)
    })

# Add 2 anomalies
rows.append({"timestamp": "2024-07-01", "temperature": 65.0, "windspeed": 5.0})
rows.append({"timestamp": "2024-07-02", "temperature": 10.0, "windspeed": 95.0})

df_anomaly = pd.DataFrame(rows)

model = IsolationForest(contamination=0.1, random_state=42)
df_anomaly['anomaly_flag'] = model.fit_predict(df_anomaly[['temperature','windspeed']])
df_anomaly['status'] = df_anomaly['anomaly_flag'].apply(
    lambda x: 'Unusual' if x == -1 else 'Normal'
)

save_to_db(df_anomaly, "weather_anomalies")
print("✅ Anomaly detection done!")
df_anomaly[['timestamp','temperature','windspeed','status']]

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Real world: Weather app predicts next 7 days
# We: Predict next 7 days temperature for Roorkee

temp_series = df_anomaly['temperature'].iloc[:30].reset_index(drop=True)
model = ExponentialSmoothing(temp_series, trend='add').fit()
forecast = model.forecast(7)

df_forecast = pd.DataFrame({
    "day": [f"Day +{i}" for i in range(1, 8)],
    "forecasted_temp": forecast.round(1).values
})

save_to_db(df_forecast, "forecast")
print("✅ Forecast done!")
df_forecast

In [ ]:
import plotly.express as px

# Chart 1 - Anomalies
fig1 = px.scatter(
    df_anomaly, x='timestamp', y='temperature',
    color='status', size='windspeed',
    title='🌡️ Temperature — Anomalies Highlighted',
    color_discrete_map={'Normal':'#2196F3','Unusual':'#F44336'}
)
fig1.write_html("output/charts/anomalies.html")
fig1.show()

# Chart 2 - Sentiment
fig2 = px.pie(
    df_complaints, names='sentiment_label',
    title='📝 Complaint Sentiments',
    color_discrete_sequence=['#4CAF50','#F44336','#9E9E9E']
)
fig2.write_html("output/charts/sentiment.html")
fig2.show()

# Chart 3 - Air Quality
fig3 = px.bar(
    df_air, x='city', y=['pm2_5','pm10'],
    title='💨 Air Quality — Roorkee',
    barmode='group'
)
fig3.write_html("output/charts/air_quality.html")
fig3.show()

# Chart 4 - Forecast
fig4 = px.line(
    df_forecast, x='day', y='forecasted_temp',
    title='📈 7-Day Temperature Forecast',
    markers=True
)
fig4.write_html("output/charts/forecast.html")
fig4.show()

print("✅ All charts saved in output/charts/")

In [ ]:
import os

os.chdir("C:/Users/DISHA SHARMA/projects/city-smart-pulse")
print("✅ Working directory:", os.getcwd())

In [ ]:
os.makedirs("data", exist_ok=True)
os.makedirs("output/charts", exist_ok=True)
print("✅ Folders ready")

In [ ]:
import sqlite3
import pandas as pd

# Test if DB is working
conn = sqlite3.connect("data/city_pulse.db")
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
conn.close()

print("✅ Database found!")
print("Tables inside:", tables)

In [ ]:
conn = sqlite3.connect("data/city_pulse.db")

try:
    with pd.ExcelWriter("data/city_pulse_data.xlsx", engine='openpyxl') as writer:
        
        for table, sheet in [
            ("weather_logs", "Weather"),
            ("air_quality_logs", "AirQuality"),
            ("complaints_analysis", "Complaints"),
            ("weather_anomalies", "Anomalies"),
            ("forecast", "Forecast")
        ]:
            try:
                df = pd.read_sql(f"SELECT * FROM {table}", conn)
                df.to_excel(writer, sheet_name=sheet, index=False)
                print(f"✅ {sheet} sheet saved — {len(df)} rows")
            except Exception as e:
                print(f"⚠️ Skipped {sheet}: {e}")

    print("\n✅ Excel file saved at: data/city_pulse_data.xlsx")

except Exception as e:
    print("❌ Error:", e)

conn.close()

In [7]:
import sqlite3
import pandas as pd

# Check what columns exist right now
conn = sqlite3.connect("data/city_pulse.db")
df_check = pd.read_sql("SELECT * FROM air_quality_logs", conn)
conn.close()

print("Current columns:", df_check.columns.tolist())
print(df_check)

Current columns: ['timestamp', 'pm2_5', 'pm10', 'carbon_monoxide', 'nitrogen_dioxide', 'city', 'aqi', 'aqi_level']
             timestamp  pm2_5   pm10  carbon_monoxide  nitrogen_dioxide  \
0  2026-06-20 15:25:48   87.3  106.9            632.0              46.8   

      city  aqi   aqi_level  
0  Roorkee  191  Moderate 🟠  
